# Mapping consortium metabolite annotations to the Human1 metabolic network

This notebook maps Level 1 metabolite annotations from the consortium pilot study onto a filtered compound graph derived from the Human1 genome-scale metabolic model. The analysis is organized into three stages:

1. harmonization of metabolite identifiers and mapping to Human1;
2. topological characterization of the mapped metabolites across devices and laboratories;
3. evaluation of network coverage using the Hamiltonian-based energy model.

The notebook is intended to document the complete analysis workflow, from the original annotation table to the final coverage estimates and network visualizations. Unless otherwise stated, metabolites are compared through ChEBI identifiers and represented as nodes in an undirected, unweighted compound graph.

## 1. Data import and identifier harmonization

The first stage loads the consortium annotation table, the original Human1 model, and the previously filtered Human1 compound graph. The original model is used to recover metabolite identifiers, whereas the filtered graph is used for the topological analyses.

Project paths are defined relative to the notebook location to make the workflow reproducible within the repository structure. The analysis uses:

- the consortium pilot-study annotation table;
- the Human1 SBML/XML model;
- the filtered Human1 compound graph stored in GML format.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from etc.parse_ids import XMLParser

# Get the notebook's directory and go up to project root
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
data_folder = project_root / "data" / "resources"
Pilot = data_folder / "PilotStudy_All"

# Get graph prevouly fitered
graph_gml = data_folder / "generated" / "modified_graph.gml"
# Get the original graph for matching ids
human1_xml = data_folder / "Human-GEM.xml"

### 1.1 Standardizing ChEBI identifiers

ChEBI identifiers in the annotation table are stored as strings such as `CHEBI:12345`. To compare them with the identifiers extracted from Human1, the numerical part of each valid ChEBI identifier is isolated and stored as an integer.


In [2]:
# Auxiliar function to extract the chebi number from a string like "CHEBI:12345"
def extract_single_chebi(value):
    """Extract CHEBI number from a single CHEBI string"""
    if pd.isna(value):
        return None
    match = re.search(r"CHEBI:(\d+)", str(value))
    if match:
        return int(match.group(1))
    return None

### 1.2 Loading the consortium pilot-study compilation

The pilot-study spreadsheet contains the metabolite annotations compiled across the analytical devices and participating laboratories. At this stage, the complete table is loaded without filtering so that the total number of reported annotations can be documented.

In [3]:
# Load the data file as a pandas DataFrame
data_file = pd.read_excel(Pilot / "Daniel_Suplementary_info.xlsx", sheet_name="Sheet1")
print(f'Total annotated {data_file.CHEBI_ID_Step2.unique().shape[0]} metabolites')

Total annotated 358 metabolites


### 1.3 Restricting the analysis to Level 1 annotations

Only metabolites assigned annotation level 1 are retained. These annotations have the highest identification confidence and therefore provide the most defensible input for comparison with the Human1 reconstruction.

After filtering, 228 distinct Level 1 metabolites remain.

In [4]:
# Filter to only include rows where ID_level is 1
data_file = data_file[data_file.ID_level==1]
print(f'Total annotated {data_file.CHEBI_ID_Step2.unique().shape[0]} metabolites level 1')

Total annotated 228 metabolites level 1


### 1.4 Creating a normalized ChEBI column

A new column containing only the numerical ChEBI identifier is added to the filtered annotation table. 

In [5]:
# Add new column with extracted CHEBI numbers
data_file['CHEBI_number'] = data_file.CHEBI_ID_Step2.apply(extract_single_chebi)

### From HUMAN1 GEM xml file 
- Create a dataframe
- Create colum to match chebis ids and link them with human1 ids

In [6]:
parser = XMLParser(human1_xml)
df = parser.extract_data()
df_human1 = parser.to_identifier_df()
df_human1['Consortium'] = None

### 1.5 Extracting Human1 metabolite identifiers

The Human1 XML model is parsed to recover the metabolite metadata and construct an identifier table. Each row represents a Human1 metabolite entry and may contain database cross-references such as ChEBI identifiers.

A Boolean `Consortium` column is initialized to indicate whether each Human1 entry can be associated with at least one Level 1 metabolite reported in the consortium dataset.

In [7]:
# Convert df_human1 CHEBI to integers for matching
df_human1['chebi_int'] = pd.to_numeric(df_human1['chebi'], errors='coerce').astype('Int64')
# Get set of CHEBIs from data_file
data_file_chebis = set(data_file['CHEBI_number'].dropna().values)

# Fill Consortium column with boolean: True if CHEBI matches, False otherwise
df_human1['Consortium'] = df_human1['chebi_int'].isin(data_file_chebis)

print(f"Consortium column filled with boolean values")
print(f"True matches: {df_human1[df_human1['Consortium'] == True].chebi.unique().shape[0]}")
print(f"False (no match): {(~df_human1['Consortium']).sum()}")


Consortium column filled with boolean values
True matches: 105
False (no match): 8100


In [8]:
# Find CHEBIs in data_file that are NOT in df_human1
df_human1_chebis = set(df_human1['chebi_int'].dropna().values)
data_file_chebis_set = set(data_file['CHEBI_number'].dropna().values)

# CHEBIs not found in df_human1
unmatched_chebis = data_file_chebis_set - df_human1_chebis

print(f"Total CHEBIs in data_file: {len(data_file_chebis_set)}")
print(f"CHEBIs found in df_human1: {len(data_file_chebis_set - unmatched_chebis)}")
print(f"CHEBIs NOT found in df_human1: {len(unmatched_chebis)}")
print(f"\nUnmatched CHEBIs: {sorted(unmatched_chebis)}")

Total CHEBIs in data_file: 228
CHEBIs found in df_human1: 105
CHEBIs NOT found in df_human1: 123

Unmatched CHEBIs: [np.int64(4139), np.int64(4828), np.int64(9008), np.int64(11901), np.int64(14314), np.int64(15620), np.int64(15761), np.int64(16020), np.int64(16113), np.int64(16119), np.int64(16231), np.int64(16312), np.int64(16347), np.int64(16373), np.int64(16831), np.int64(17012), np.int64(17016), np.int64(17072), np.int64(17234), np.int64(17597), np.int64(17687), np.int64(17780), np.int64(18123), np.int64(19030), np.int64(19062), np.int64(19065), np.int64(19660), np.int64(21264), np.int64(21553), np.int64(21563), np.int64(21756), np.int64(21949), np.int64(25858), np.int64(27410), np.int64(27596), np.int64(27732), np.int64(27747), np.int64(27838), np.int64(28123), np.int64(28177), np.int64(28238), np.int64(28393), np.int64(28664), np.int64(28717), np.int64(28775), np.int64(28821), np.int64(28871), np.int64(28946), np.int64(30753), np.int64(30776), np.int64(30845), np.int64(35280), np

note: 123 metbolites from the data couldn't be match on HUMAN1, note some are lipis and most of them considered sidecompounds that will not be taken into account in the model. Also we only included level 1 annotated metabolites

### Mask separated annotations on HUMAN1 dataframe
- For different devices
- For different labs

In [9]:
capitainer = data_file[data_file.Capitainer == "Yes"]
mitra = data_file[data_file.Mitra == "Yes"]
blood = data_file[data_file.Blood == "Yes"]
plasma = data_file[data_file.Plasma == "Yes"]

devices = {
    "Capitainer": capitainer,
    "Mitra": mitra,
    "Blood": blood,
    "Plasma": plasma
}
for device_name, device_df in devices.items():
    chebis = set(device_df['CHEBI_number'].dropna().values)
    df_human1[device_name] = df_human1['chebi_int'].isin(device_df['CHEBI_number'].dropna().values)

    print(f"{device_name} column filled with boolean values")
    print(f"True matches: {df_human1[device_name].sum()}")
    print(f"False (no match): {(~df_human1[device_name]).sum()}")

Capitainer column filled with boolean values
True matches: 332
False (no match): 8124
Mitra column filled with boolean values
True matches: 338
False (no match): 8118
Blood column filled with boolean values
True matches: 321
False (no match): 8135
Plasma column filled with boolean values
True matches: 330
False (no match): 8126


In [10]:
hmgu = data_file[data_file.HMGU == 1]
cembio = data_file[data_file.CEMBIO == 1]
afekta = data_file[data_file.AFEKTA == 1]
icl = data_file[data_file.ICL == 1]
auth = data_file[data_file.AUTh == 1]

labs = {
    "HMGU": hmgu,
    "CEMBIO": cembio,
    "AFEKTA": afekta,
    "ICL": icl,
    "AUTh": auth
    }
for lab_name, lab_df in labs.items():
    chebis = set(lab_df['CHEBI_number'].dropna().values)
    df_human1[lab_name] = df_human1['chebi_int'].isin(lab_df['CHEBI_number'].dropna().values)

    print(f"{lab_name} column filled with boolean values")
    print(f"True matches: {df_human1[lab_name].sum()}")
    print(f"False (no match): {(~df_human1[lab_name]).sum()}")

HMGU column filled with boolean values
True matches: 122
False (no match): 8334
CEMBIO column filled with boolean values
True matches: 283
False (no match): 8173
AFEKTA column filled with boolean values
True matches: 140
False (no match): 8316
ICL column filled with boolean values
True matches: 116
False (no match): 8340
AUTh column filled with boolean values
True matches: 150
False (no match): 8306


## 2. Mapping the annotations onto the filtered Human1 graph

The topological analysis is performed on a filtered compound graph derived from Human1. Nodes represent metabolites, and edges connect metabolites associated through the same reaction after the graph preprocessing steps.

The graph used in this notebook contains 3,240 nodes and 7,254 edges and forms a single connected component. Node order is stored explicitly so that Human1 metabolite identifiers can be converted into integer positions compatible with the Hamiltonian implementation.

In [11]:
import networkx as nx
from matplotlib import pyplot as plt
from upsetplot import UpSet, from_memberships

graph = nx.read_gml(graph_gml)
n = graph.number_of_nodes()
graph_nodes = list(graph.nodes())
node_to_index = {node: idx for idx, node in enumerate(graph_nodes)}

print(f"Graph loaded with {n} nodes.")
print(f"Graph has {graph.number_of_edges()} edges.")
print(f"Graph is connected: {nx.is_connected(graph)}")

Graph loaded with 3240 nodes.
Graph has 7254 edges.
Graph is connected: True


### 2.1 Mapping device-specific metabolites to graph nodes

For each device, matched Human1 metabolite identifiers are intersected with the nodes retained in the filtered graph.

In [12]:
device_indices = {}

for device_name in ['Mitra', 'Capitainer', 'Blood', 'Plasma']:
    device_filtered = df_human1[df_human1[device_name] == True]
    device_ids = set(device_filtered.index.tolist())

    selected_indices = [node_to_index[node_id] for node_id in device_ids if node_id in node_to_index]
    device_indices[device_name] = selected_indices
    print(f"{device_name} indices mapped to graph nodes: {len(selected_indices)} found.")

Mitra indices mapped to graph nodes: 77 found.
Capitainer indices mapped to graph nodes: 75 found.
Blood indices mapped to graph nodes: 70 found.
Plasma indices mapped to graph nodes: 73 found.


### 2.2 Overlap between analytical devices

An UpSet plot is used to summarize intersections between the device-specific metabolite sets. Each bar represents the number of graph nodes observed by a particular combination of devices.

This representation distinguishes three different situations:

- metabolites shared by all or most devices;
- metabolites detected by a restricted subset of devices;
- metabolites unique to one device.

The plot therefore describes annotation overlap, but it does not yet determine whether non-overlapping metabolites occupy distinct or similar regions of the metabolic network.

In [ ]:
# Build membership list for each observed graph node
all_nodes = sorted(set().union(*device_indices.values()))
memberships = [
    tuple(sorted([device for device, indices in device_indices.items() if node_idx in indices]))
    for node_idx in all_nodes
]

upset_data = from_memberships(memberships)

plt.figure(figsize=(10, 6))
upset = UpSet(upset_data, subset_size='count', show_counts=True, sort_by='degree')
upset.plot()
plt.title('Device intersections for observed metabolites')
plt.tight_layout()
plt.savefig(data_folder / "figs" / "device_intersections_upset.png")
plt.show()

### 2.3 Mapping laboratory-specific metabolites to graph nodes

The same mapping procedure is applied to the laboratory-specific annotation sets. 

In [13]:
lab_indices = {}
for lab_name in ['HMGU', 'CEMBIO', 'AFEKTA', 'ICL', 'AUTh']:
    lab_filtered = df_human1[df_human1[lab_name] == True]
    lab_ids = set(lab_filtered.index.tolist())

    selected_indices = [node_to_index[node_id] for node_id in lab_ids if node_id in node_to_index]
    lab_indices[lab_name] = selected_indices
    print(f"{lab_name} indices mapped to graph nodes: {len(selected_indices)} found.")

HMGU indices mapped to graph nodes: 25 found.
CEMBIO indices mapped to graph nodes: 57 found.
AFEKTA indices mapped to graph nodes: 24 found.
ICL indices mapped to graph nodes: 27 found.
AUTh indices mapped to graph nodes: 21 found.


### 2.4 Overlap between laboratories

A second UpSet plot summarizes the intersections among laboratory-specific metabolite sets.

In [ ]:
# Build membership list for each observed graph node
all_lab_nodes = sorted(set().union(*lab_indices.values()))
lab_memberships = [
    tuple(sorted([lab for lab, indices in lab_indices.items() if node_idx in indices]))
    for node_idx in all_lab_nodes]
lab_upset_data = from_memberships(lab_memberships)
plt.figure(figsize=(10, 6))
lab_upset = UpSet(lab_upset_data, subset_size='count', show_counts=True, sort_by='degree')
lab_upset.plot()
plt.title('Lab intersections for observed metabolites')
plt.tight_layout()
plt.savefig(data_folder / "figs" / "lab_intersections_upset.png")
plt.show()

## 3. Topological characterization of observed metabolites

### 3.1 Degree distribution of device-specific observations

The graph degree distribution is compared with the degree distributions of metabolites observed by each device. 

In [ ]:
degrees = [d for _, d in graph.degree()]
unique, counts = np.unique(degrees, return_counts=True)

degree_devices = {
    device_name: np.array(degrees)[device_indices[device_name]] for device_name in devices.keys()
    }
freq = counts / counts.sum()

plt.figure(figsize=(6,4))
plt.loglog(unique, freq, 'o', markersize=4, label='all degrees', alpha=0.8)
for device_name, device_degrees in degree_devices.items():
    unique_dev, counts_dev = np.unique(device_degrees, return_counts=True)
    freq_dev = counts_dev / counts_dev.sum()
    plt.loglog(unique_dev, freq_dev, 'o', markersize=6, label=device_name, alpha=0.8)
plt.xlabel('Degree (k)')
plt.ylabel('Frequency P(k)')
plt.title('Degree Distribution of Graph and Devices')
plt.legend()
plt.grid(True, which="both", ls="--", linewidth=0.5)

plt.savefig(data_folder / 'figs' / "degree_distribution.png", dpi=300, bbox_inches='tight')
plt.show()

### 3.2 Degree distribution of laboratory-specific observations

The same degree-based comparison is performed for each participating laboratory.

In [ ]:
degree_labs = {
    lab_name: np.array(degrees)[lab_indices[lab_name]] for lab_name in labs.keys()
    }
plt.figure(figsize=(6,4))
plt.loglog(unique, freq, 'o', markersize=4, label='all degrees', alpha=0.8)
for lab_name, lab_degrees in degree_labs.items():
    unique_lab, counts_lab = np.unique(lab_degrees, return_counts=True)
    freq_lab = counts_lab / counts_lab.sum()
    plt.loglog(unique_lab, freq_lab, 'o', markersize=6, label=lab_name, alpha=0.8)
plt.xlabel('Degree (k)')
plt.ylabel('Frequency P(k)')
plt.title('Degree Distribution of Graph and Labs')
plt.legend()
plt.grid(True, which="both", ls="--", linewidth=0.5)
plt.savefig(data_folder / 'figs' / "degree_distribution_labs.png", dpi=300, bbox_inches='tight')
plt.show()

### 3.3 Connector subgraph spanning the consortium observations

In [ ]:
observed_nodes = [
    n for n in df_human1.index
    if df_human1.loc[n, "Consortium"] and n in graph
    ]
observed_indices = [node_to_index[node_id] for node_id in observed_nodes]

connector_indices = set(observed_indices)

for i, source_node in enumerate(observed_nodes):
    for target_node in observed_nodes[i + 1 :]:
        try:
            path = nx.shortest_path(graph, source=source_node, target=target_node)
            connector_indices.update(node_to_index[node_id] for node_id in path)
        except nx.NetworkXNoPath:
            pass

print(f"Total observed nodes (including connectors): {len(connector_indices)}")
connector_nodes = [graph_nodes[idx] for idx in connector_indices]

G_observed = graph.subgraph(connector_nodes).copy()
print(G_observed)
print(f"Nodes: {G_observed.number_of_nodes()}")
print(f"Edges: {G_observed.number_of_edges()}")

### 3.4 Geometric Voronoi representation of observed regions

A spring layout assigns two-dimensional coordinates to all graph nodes, and a geometric Voronoi tessellation is constructed around the union of device-observed metabolites. Each Voronoi region contains the points that are geometrically closest to one observed metabolite in the chosen two-dimensional layout.

This figure provides an intuitive visualization of the spatial influence of the observations in the layout. It must not be interpreted as an exact graph-theoretical partition: Euclidean distances in a spring layout do not equal shortest-path distances in the metabolic network. The Voronoi diagram is therefore used only as a qualitative visualization of the apparent spread of observed metabolites.#### Voronoi graph regions and observations

In [ ]:
from scipy.spatial import Voronoi, voronoi_plot_2d

# 1. Get 2D coordinates for all graph nodes
pos = nx.spring_layout(graph, seed=42, iterations=200)

# 2. Observed nodes = union across devices
observed_nodes = sorted(set().union(*[set(v) for v in device_indices.values()]))
observed_nodes = [n for n in observed_nodes if n in graph]

points = np.array([pos[n] for n in observed_nodes])

# 1. Get 2D coordinates for all graph nodes
pos = nx.spring_layout(graph, seed=42, iterations=200)

# 2. Observed nodes = union across devices
observed_indices = sorted(set().union(*[set(v) for v in device_indices.values()]))
observed_nodes = [graph_nodes[idx] for idx in observed_indices if idx < len(graph_nodes)]
points = np.array([pos[n] for n in observed_nodes])

if points.ndim != 2 or points.shape[0] < 2:
    raise ValueError(f"Need at least 2 observed points for Voronoi, got {points.shape}")

# 3. Compute Voronoi from observed metabolites
vor = Voronoi(points)

# 4. Plot Voronoi regions
fig, ax = plt.subplots(figsize=(10, 10))

voronoi_plot_2d(
    vor,
    ax=ax,
    show_vertices=False,
    show_points=False,
    line_width=0.6,
    line_alpha=0.8
)

# 5. Plot all Human1 nodes in background
all_xy = np.array([pos[n] for n in graph.nodes()])
ax.scatter(
    all_xy[:, 0],
    all_xy[:, 1],
    s=3,
    alpha=0.25,
    color="gray"
)

# 6. Plot observed metabolites
ax.scatter(
    points[:, 0],
    points[:, 1],
    s=18,
    color="black",
    zorder=5
)

ax.set_title("Geometric Voronoi regions around observed metabolites")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(data_folder / 'figs' / 'voronoi_plot_2d.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Energy-based model of topological coverage
The mapped metabolite sets are evaluated using a Hamiltonian that combines local biochemical coherence and global topological dispersion.

For an observation vector $\mathbf{s}$, where $s_i=1$ when metabolite $i$ is observed and $s_i=0$ otherwise, the $H$ function is

$$
\mathcal{H}(\mathbf{s})
=
-\mu\sum_{i<j}A_{ij}s_is_j
+
\gamma\sum_{i<j}
\frac{(1-A_{ij})s_is_j}{d_{ij}^{2}},
$$

where $A_{ij}$ is the adjacency matrix and $d_{ij}$ is the shortest-path distance between nodes $i$ and $j$.

The first term,

$$
T_1=-\mu\sum_{i<j}A_{ij}s_is_j,
$$

rewards direct adjacency among observed metabolites and represents local biochemical coherence. The second term,

$$
T_2=\gamma\sum_{i<j}
\frac{(1-A_{ij})s_is_j}{d_{ij}^{2}},
$$

penalizes non-adjacent observations that remain topologically close and therefore promotes broader dispersion across the network.

The effective energy used for coverage is

$$
E(\mathbf{s})=\left|\mathcal{H}(\mathbf{s})\right|.
$$

Configurations with $\mathcal{H}<0$ are attraction-dominated, whereas configurations with $\mathcal{H}>0$ are repulsion-dominated. Values near zero indicate a balance between local connectivity and global spread.


In [14]:
from etc.hamiltonian import Hamiltonian
from etc.optimization import EnergyOptimizer

### 4.1 Model initialization and graph-dependent parameters

A `Hamiltonian` object is constructed from the filtered Human1 graph, together with an `EnergyOptimizer` used for random sampling and simulated annealing.

The initial values of $\mu$ and \(\gamma\) are obtained from graph-dependent helper methods. For the present graph, the density-aware procedure gives approximately

$$
\mu = 0.9986,
\qquad
\gamma = 2.5562.
$$

These values provide an initial graph-scaled parameterization rather than parameters inferred directly from the experimental observations.

In [28]:
H = Hamiltonian(graph)
optimizer = EnergyOptimizer(H)
mu = H.mu_density_aware(graph)
gamma = H.gamma_balancer(mu=mu)

print(f"Gamma value set to: {gamma:.4f}")
print(f"Mu value set to: {mu:.4f}")

Gamma value set to: 2.5562
Mu value set to: 0.9986


### 4.2 Adjustment of the dispersion parameter

For the empirical comparison, $\gamma$ is reassigned using

$$
\gamma=(1-\mu) * \overline{d}^2,
$$

which gives approximately

$$
\gamma=0.0677.
$$

In [30]:
gamma = (1-mu)*7*7
print(f"Gamma value set to: {gamma:.4f}")

Gamma value set to: 0.0677


In [17]:
# Average shortest path length of the graph
avg_shortest_path_length = nx.average_shortest_path_length(graph)
print(f"Average shortest path length of the graph: {avg_shortest_path_length:.4f}")

Average shortest path length of the graph: 7.3920


In [18]:
distance_matrix = nx.floyd_warshall_numpy(graph)

## 5. H values of the empirical configurations

The $H$ function and its two components are computed for every device- and laboratory-specific metabolite set using the selected values of $\mu$ and $\gamma$.

Reporting $\mathcal{H}$, $T_1$, and $T_2$ separately 


In [31]:
device_h_values = {}
for device_name in devices.keys():
    h,t1,t2 = H.compute(device_indices[device_name], gamma=gamma, mu=mu)
    device_h_values[device_name] = (h, t1, t2)
    
print("H values computed for each device:")
for device_name, values in device_h_values.items():
    print(f"{device_name}: H={values[0]:.4f}, t1={values[1]:.4f}, t2={values[2]:.4f}, k={len(device_indices[device_name])}")
print(f"gamma = {gamma:.4f}, mu = {mu:.4f}")

H values computed for each device:
Capitainer: H=-32.6563, t1=-41.9419, t2=9.2857, k=75
Mitra: H=-34.9921, t1=-44.9378, t2=9.9457, k=77
Blood: H=-28.1786, t1=-35.9502, t2=7.7716, k=70
Plasma: H=-27.4899, t1=-35.9502, t2=8.4603, k=73
gamma = 0.0677, mu = 0.9986


In [20]:
labs_h_values = {}
for lab_name in labs.keys():
    h,t1,t2 = H.compute(lab_indices[lab_name], gamma=gamma, mu=mu)
    labs_h_values[lab_name] = (h, t1, t2)
print("H values computed for each lab:")
for lab_name, values in labs_h_values.items():
    print(f"{lab_name}: H={values[0]:.4f}, t1={values[1]:.4f}, t2={values[2]:.4f}, k={len(lab_indices[lab_name])}")
print(f"gamma = {gamma:.4f}, mu = {mu:.4f}")

H values computed for each lab:
HMGU: H=-4.9064, t1=-5.9917, t2=1.0853, k=25
CEMBIO: H=-20.6937, t1=-25.9641, t2=5.2703, k=57
AFEKTA: H=-3.0069, t1=-3.9945, t2=0.9876, k=24
ICL: H=-3.1048, t1=-3.9945, t2=0.8896, k=27
AUTh: H=-2.1276, t1=-2.9959, t2=0.8683, k=21
gamma = 0.0677, mu = 0.9986


## 6. Random reference distributions

In [21]:
energies_devices = {}
for device_name in devices.keys():
    energies = optimizer.sampling_h(
        n=n,k=len(device_indices[device_name]), 
        gamma=gamma, mu=mu,
        n_samples=1000,n_workers=8,seed=42)[0]
    print(f"{device_name} Min energy: {energies.min()}, Max energy: {energies.max()}")
    energies_devices[device_name] = energies

Capitainer Min energy: -7.348731588200815, Max energy: 7.453051173319189
Mitra Min energy: -9.582567371615463, Max energy: 8.498410594930942
Blood Min energy: -7.5422211550575975, Max energy: 6.0479026325756715
Plasma Min energy: -7.444136406218287, Max energy: 6.77583675277136


In [22]:
energies_labs = {}
for lab_name in labs.keys():
    energies = optimizer.sampling_h(
        n=n,k=len(lab_indices[lab_name]), 
        gamma=gamma, mu=0.9,
        n_samples=1000,n_workers=8,seed=42)[0]
    print(f"{lab_name} Min energy: {energies.min()}, Max energy: {energies.max()}")
    energies_labs[lab_name] = energies

HMGU Min energy: -4.361170491160485, Max energy: 1.0230386841824588
CEMBIO Min energy: -6.171475610569372, Max energy: 4.268472909206929
AFEKTA Min energy: -2.1524292389485504, Max energy: 0.9544051334611316
ICL Min energy: -3.0256650912617395, Max energy: 1.2610815816213168
AUTh Min energy: -2.82104732807879, Max energy: 0.8360889877764329


## 7. Maximum-energy estimation

The maximum effective energy is estimated by simulated annealing. Because widely separated observations are expected to generate a strongly repulsion-dominated configuration, the initial state for each search is constructed with the existing farthest-node sampler.

A separate starting configuration is generated for every device and laboratory using its corresponding value of $k$. This ensures that the normalization reference is cardinality-specific.

### 7.1 Device-specific maximum-energy initializations

Farthest-node configurations are generated for the four device sizes:

In [23]:
S0Capitainer = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=75, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0Mitra = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=77, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0Blood = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=70, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0Plasma = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=73, seed=42,distance_matrix=distance_matrix),
    graph.nodes())


### 7.2 Laboratory-specific maximum-energy initializations

In [24]:
S0HMGU = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=25, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0CEMBIO = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=57, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0AFEKTA = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=24, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0ICL = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=27, seed=42,distance_matrix=distance_matrix),
    graph.nodes())
S0AUTh = optimizer.create_S0_mask(
    optimizer.farthest_nodes_sample(graph=graph, k=21, seed=42,distance_matrix=distance_matrix),
    graph.nodes())

In [25]:
# Dictionary of configurations: (S0_config, gamma, name)
configs = {
    'cap': (S0Capitainer, gamma),
    'mitra': (S0Mitra, gamma),
    'blood': (S0Blood, gamma),
    'plasma': (S0Plasma, gamma),
    'hmgu': (S0HMGU, gamma),
    'afekta': (S0AFEKTA, gamma),
    'cembio': (S0CEMBIO, gamma),
    'icl': (S0ICL, gamma),
    'auth': (S0AUTh, gamma),
}

# Compute max energy for all configurations
emax_values = {}
for name, (S0_config, gamma) in configs.items():
    emax_values[f'Emax_{name}'] = optimizer.max_energy_annealing(
        S0_config, gamma=gamma, mu=mu, steps=10000, n_workers=8, Tmin=1e-60
    )[1]

# Extract individual variables for backward compatibility
print(f'emax_values[\'Emax_capitainer\'] = {emax_values["Emax_cap"]}')
print(f'emax_values[\'Emax_mitra\'] = {emax_values["Emax_mitra"]}')
print(f'emax_values[\'Emax_blood\'] = {emax_values["Emax_blood"]}')
print(f'emax_values[\'Emax_plasma\'] = {emax_values["Emax_plasma"]}')
print(f'emax_values[\'Emax_hmgu\'] = {emax_values["Emax_hmgu"]}')
print(f'emax_values[\'Emax_afekta\'] = {emax_values["Emax_afekta"]}')
print(f'emax_values[\'Emax_cembio\'] = {emax_values["Emax_cembio"]}')
print(f'emax_values[\'Emax_icl\'] = {emax_values["Emax_icl"]}')
print(f'emax_values[\'Emax_auth\'] = {emax_values["Emax_auth"]}')

print("All Emax values computed successfully")

emax_values['Emax_capitainer'] = 95.40965701310242
emax_values['Emax_mitra'] = 105.03096168951066
emax_values['Emax_blood'] = 102.0607325539971
emax_values['Emax_plasma'] = 71.31446131382657
emax_values['Emax_hmgu'] = 48.65568371120227
emax_values['Emax_afekta'] = 65.24042500344379
emax_values['Emax_cembio'] = 69.1613901009877
emax_values['Emax_icl'] = 60.7368147442764
emax_values['Emax_auth'] = 47.65873939125615
All Emax values computed successfully


## 8. Normalized topological coverage

The effective energy is converted into a normalized coverage score:

$$
C(\mathbf{s})
=
1-
\frac{E(\mathbf{s})-E_{\min}}
     {E_{\max}-E_{\min}}.
$$

In the current implementation,

$$
E_{\min}=0,
$$

corresponding to the ideal balance condition $\mathcal{H}=0$. The normalization therefore simplifies to

$$
C(\mathbf{s})
=
1-\frac{|\mathcal{H}(\mathbf{s})|}{E_{\max}}.
$$

A score close to one indicates that the configuration lies near the balance between local biochemical coherence and global topological dispersion relative to the estimated maximum energy for the same $k$. A lower score indicates a stronger deviation toward either excessive clustering or excessive dispersion.

The score is topology-aware but remains conditional on the selected graph representation, parameterization, and numerical estimate of $E_{\max}$.


In [26]:
def coverage(E,Emax,Emin:int=0): 
    return 1- (E-Emin)/(Emax-Emin)

In [27]:
print(f'C(Capitainer) = {coverage(E=abs(device_h_values["Capitainer"][0]),Emax=emax_values["Emax_cap"] )}')
print(f'C(Mitra) = {coverage(E=abs(device_h_values["Mitra"][0]),Emax=emax_values["Emax_mitra"] )}')
print(f'C(Blood) = {coverage(E=abs(device_h_values["Blood"][0]),Emax=emax_values["Emax_blood"] )}')
print(f'C(Plasma) = {coverage(E=abs(device_h_values["Plasma"][0]),Emax=emax_values["Emax_plasma"] )}')
print(f'C(HMGU) = {coverage(E=abs(labs_h_values["HMGU"][0]),Emax=emax_values["Emax_hmgu"] )}')
print(f'C(AFEKTA) = {coverage(E=abs(labs_h_values["AFEKTA"][0]),Emax=emax_values["Emax_afekta"] )}')
print(f'C(CEMBIO) = {coverage(E=abs(labs_h_values["CEMBIO"][0]),Emax=emax_values["Emax_cembio"] )}')
print(f'C(ICL) = {coverage(E=abs(labs_h_values["ICL"][0]),Emax=emax_values["Emax_icl"] )}')
print(f'C(AUTh) = {coverage(E=abs(labs_h_values["AUTh"][0]),Emax=emax_values["Emax_auth"] )}')

C(Capitainer) = 0.6577259465192851
C(Mitra) = 0.6668403640689244
C(Blood) = 0.7239031787789598
C(Plasma) = 0.6145251998084758
C(HMGU) = 0.8991609713807572
C(AFEKTA) = 0.953910183579281
C(CEMBIO) = 0.7007905404515348
C(ICL) = 0.948880480872913
C(AUTh) = 0.9553580624599007


## 9. Network visualization of laboratory observations

The final part of the notebook visualizes the laboratory-specific observations within shared local subgraphs of Human1. These figures complement the scalar coverage score by showing whether laboratory annotations occupy overlapping neighborhoods or different regions of the network.

Observed nodes are mapped back from integer positions to Human1 graph-node identifiers before visualization.

In [ ]:
from matplotlib.lines import Line2D

def plot_lab_subgraph_regions(
    g,
    lab_indices,
    graph_nodes,
    *,
    layout="spring",
    seed=42,
    node_size=80,
    observed_size=260,
    alpha_edges=0.25,
):
    """
    Plot a subgraph and highlight observed metabolites by lab.

    Parameters
    ----------
    g : nx.Graph
        Subgraph to visualize.
    lab_indices : dict
        Dictionary {lab_name: list_of_node_indices}.
    graph_nodes : list
        List mapping integer indices to graph node names.
    """

    # Convert lab indices to node names and keep only nodes inside g
    lab_nodes = {
        lab: {
            graph_nodes[idx]
            for idx in indices
            if idx < len(graph_nodes) and graph_nodes[idx] in g
        }
        for lab, indices in lab_indices.items()
    }

    observed_nodes = set().union(*lab_nodes.values()) if lab_nodes else set()

    # Layout
    if layout == "spring":
        pos = nx.spring_layout(g, seed=seed, iterations=300)
    elif layout == "kamada":
        pos = nx.kamada_kawai_layout(g)
    else:
        raise ValueError("layout must be 'spring' or 'kamada'")

    colors = {
        "HMGU": "tab:blue",
        "CEMBIO": "tab:orange",
        "AFEKTA": "tab:green",
        "ICL": "tab:red",
        "AUTh": "tab:purple",
    }

    fig, ax = plt.subplots(figsize=(11, 9))

    # Draw edges
    nx.draw_networkx_edges(
        g,
        pos,
        ax=ax,
        edge_color="gray",
        alpha=alpha_edges,
        width=0.8,
    )

    # Draw all background nodes
    background_nodes = [n for n in g.nodes if n not in observed_nodes]
    nx.draw_networkx_nodes(
        g,
        pos,
        nodelist=background_nodes,
        node_color="lightgray",
        node_size=node_size,
        alpha=0.45,
        linewidths=0,
        ax=ax,
    )

    # Draw observed nodes lab by lab
    for lab, nodes in lab_nodes.items():
        if not nodes:
            continue

        nx.draw_networkx_nodes(
            g,
            pos,
            nodelist=list(nodes),
            node_color=colors.get(lab, "black"),
            node_size=observed_size,
            alpha=0.85,
            edgecolors="black",
            linewidths=0.8,
            label=f"{lab} ({len(nodes)})",
            ax=ax,
        )

    # Highlight nodes shared by more than one lab with larger black ring
    node_lab_count = {}
    for lab, nodes in lab_nodes.items():
        for n in nodes:
            node_lab_count[n] = node_lab_count.get(n, 0) + 1

    shared_nodes = [n for n, c in node_lab_count.items() if c > 1]

    if shared_nodes:
        nx.draw_networkx_nodes(
            g,
            pos,
            nodelist=shared_nodes,
            node_color="none",
            node_size=observed_size + 140,
            edgecolors="black",
            linewidths=2.2,
            ax=ax,
        )

    ax.set_title("Observed metabolites by institution in the selected Human1 subgraph")
    ax.set_axis_off()
    ax.legend(frameon=False, loc="best")

    plt.tight_layout()
    plt.savefig(data_folder / "figs" / "lab_subgraph_regions.png", dpi=300, bbox_inches='tight')
    plt.show()

    return lab_nodes

In [ ]:
observed_indices = set().union(*[set(v) for v in lab_indices.values()])
observed_nodes = {
    graph_nodes[idx]
    for idx in observed_indices
    if idx < len(graph_nodes) and graph_nodes[idx] in graph
}

g_nodes = set(observed_nodes)

for n in observed_nodes:
    g_nodes.update(nx.single_source_shortest_path_length(graph, n, cutoff=1).keys())

g = graph.subgraph(g_nodes).copy()

### 9.1 Laboratory-specific panels on a common graph

Each laboratory is displayed in a separate panel while retaining the same subgraph and node coordinates. In every panel:

- metabolites observed by the highlighted laboratory are shown prominently;
- metabolites observed only by other laboratories remain visible in gray;
- unobserved neighborhood nodes provide the local network context.

In [ ]:
lab_nodes_in_g = plot_lab_subgraph_regions(
    g,
    lab_indices,
    graph_nodes,
    layout="spring",
)

In [ ]:
observed = set().union(*lab_nodes_in_g.values())
g = nx.ego_graph(graph, list(observed)[0], radius=2)

In [ ]:
import math
def build_observed_neighborhood_subgraph(
    graph,
    lab_indices,
    graph_nodes,
    *,
    radius=1,
):
    """
    Build one shared subgraph containing all observed lab metabolites
    and their local graph neighborhoods.
    """

    lab_nodes = {
        lab: {
            graph_nodes[idx]
            for idx in indices
            if idx < len(graph_nodes) and graph_nodes[idx] in graph
        }
        for lab, indices in lab_indices.items()
    }

    observed_nodes = set().union(*lab_nodes.values())

    g_nodes = set(observed_nodes)

    for node in observed_nodes:
        neighbors = nx.single_source_shortest_path_length(
            graph,
            node,
            cutoff=radius,
        ).keys()
        g_nodes.update(neighbors)

    g = graph.subgraph(g_nodes).copy()

    return g, lab_nodes, observed_nodes


def plot_each_lab_on_same_subgraph(
    graph,
    lab_indices,
    graph_nodes,
    *,
    radius=1,
    layout="spring",
    seed=42,
    ncols=3,
    figsize=(16, 10),
    node_size=18,
    observed_size=100,
):
    """
    Plot one graph per lab using the same subgraph and same node positions.
    """

    g, lab_nodes, observed_nodes = build_observed_neighborhood_subgraph(
        graph,
        lab_indices,
        graph_nodes,
        radius=radius,
    )

    # Same layout for all panels
    if layout == "spring":
        pos = nx.spring_layout(g, seed=seed, iterations=300)
    elif layout == "kamada":
        pos = nx.kamada_kawai_layout(g)
    else:
        raise ValueError("layout must be 'spring' or 'kamada'")

    labs = list(lab_nodes.keys())
    n_labs = len(labs)
    nrows = math.ceil(n_labs / ncols)

    colors = {
        "HMGU": "tab:blue",
        "CEMBIO": "tab:orange",
        "AFEKTA": "tab:green",
        "ICL": "tab:red",
        "AUTh": "tab:purple",
    }

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=figsize,
        squeeze=False,
    )

    for ax, lab in zip(axes.flat, labs):
        highlighted_nodes = lab_nodes[lab]

        background_observed = observed_nodes - highlighted_nodes
        background_unobserved = set(g.nodes()) - observed_nodes

        # Edges
        nx.draw_networkx_edges(
            g,
            pos,
            ax=ax,
            edge_color="gray",
            alpha=0.18,
            width=0.6,
        )

        # Unobserved neighborhood nodes
        nx.draw_networkx_nodes(
            g,
            pos,
            nodelist=list(background_unobserved),
            node_color="lightgray",
            node_size=node_size,
            alpha=0.25,
            linewidths=0,
            ax=ax,
        )

        # Observed nodes from other labs
        nx.draw_networkx_nodes(
            g,
            pos,
            nodelist=list(background_observed),
            node_color="gray",
            node_size=node_size * 1.8,
            alpha=0.35,
            linewidths=0,
            ax=ax,
        )

        # Highlight current lab
        nx.draw_networkx_nodes(
            g,
            pos,
            nodelist=list(highlighted_nodes),
            node_color=colors.get(lab, "black"),
            node_size=observed_size,
            alpha=0.9,
            edgecolors="black",
            linewidths=0.6,
            ax=ax,
        )

        ax.set_title(f"{lab} observed metabolites (n={len(highlighted_nodes)})")
        ax.set_axis_off()

    # Hide empty panels
    for ax in axes.flat[n_labs:]:
        ax.set_visible(False)

    legend_elements = [
        Line2D(
            [0], [0],
            marker="o",
            color="w",
            markerfacecolor="lightgray",
            markersize=7,
            label="Unobserved neighborhood nodes",
        ),
        Line2D(
            [0], [0],
            marker="o",
            color="w",
            markerfacecolor="gray",
            markersize=7,
            label="Observed in other labs",
        ),
        Line2D(
            [0], [0],
            marker="o",
            color="w",
            markerfacecolor="black",
            markeredgecolor="black",
            markersize=8,
            label="Highlighted lab",
        ),
    ]

    fig.legend(
        handles=legend_elements,
        loc="lower center",
        ncol=3,
        frameon=False,
    )

    fig.suptitle(
        f"Observed metabolites by institution in the Human1 local neighborhood subgraph (radius={radius})",
        fontsize=14,
    )

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.savefig(data_folder / "figs" / "lab_subgraph_regions_all.png", dpi=300, bbox_inches='tight')
    plt.show()

    return g, lab_nodes, pos

### 9.2 Radius-one neighborhood comparison

The first multi-panel figure uses a shortest-path radius of one. It therefore emphasizes direct biochemical neighbors of the observed metabolites and provides a local view of overlap, adjacency, and immediate network context.

This radius is most suitable for examining local coherence, but it may fragment broader relationships that require intermediate connector nodes.

In [ ]:
g_labs, lab_nodes, pos = plot_each_lab_on_same_subgraph(
    graph=graph,
    lab_indices=lab_indices,
    graph_nodes=graph_nodes,
    radius=1,
    layout="spring",
    ncols=3,
    figsize=(16, 10),
)

### 9.3 Radius-two neighborhood comparison

The second multi-panel figure expands the shared subgraph to a shortest-path radius of two. The additional context reveals whether apparently separated observations are connected through short intermediate paths.

The radius-two visualization is more informative about mesoscale organization but is also denser and more difficult to interpret. Both radius-one and radius-two views are therefore retained because they emphasize different structural scales.

In [ ]:
g_labs, lab_nodes, pos = plot_each_lab_on_same_subgraph(
    graph=graph,
    lab_indices=lab_indices,
    graph_nodes=graph_nodes,
    radius=2,
    layout="spring",
    ncols=3,
    figsize=(18, 12),
)